# UNO Vision — Project Report

This notebook describes the full pipeline for detecting and classifying UNO cards from a top-down photograph of an ongoing game.
The output is a CSV describing the cards visible for each player and the center pile.

**Pipeline overview**

```
Raw image
    │
    ├── A.1  Background classification  ─────────────► 'white' or 'noisy'
    ├── A.2  Active player detection    ─────────────► p1 / p2 / p3 / p4
    ├── A.3  Background removal         ─────────────► cropped game area
    ├── A.4  Image division             ─────────────► 5 regions (4 players + center)
    │
    └── For each region:
            │
            ├── A.5  Card detection     ─────────────► list of (card_image, color)
            └── A.6  Card classification ────────────► label  (e.g. r_5, b_skip)
```

---
## Part A — Pipeline Explanation

### A.1 Background Classification (`src/background_classification.py`)

Before any card or token can be found, we must know what kind of background the photo was taken on.
The two possible backgrounds in the dataset are:

| Label | Description | Active-player token |
|-------|-------------|---------------------|
| `white` | Plain white table surface | Black rectangular tile |
| `noisy` | Patterned tablecloth or dark surface | Yellow circular disc |

This label feeds directly into A.2, which selects different HSV ranges depending on the token type.

**Method — corner brightness sampling**

UNO cards are always placed in the interior of the image; the four 50 × 50 pixel corners are reliably card-free and contain only background pixels.
We compute the mean brightness (average over all three RGB channels and all pixels) of those four patches:

$$\bar{b} = \frac{1}{4}\sum_{i=1}^{4} \text{mean}(\text{patch}_i)$$

A threshold of **200 / 255** separates the two classes:

$$\text{background} = \begin{cases} \texttt{white} & \bar{b} \ge 200 \\ \texttt{noisy} & \bar{b} < 200 \end{cases}$$

**Justification:** white surfaces reflect nearly all light and produce per-channel values close to 255, while patterned or coloured surfaces have a significantly lower mean. The fixed threshold at 200 leaves a wide margin and is robust to moderate lighting variations.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.path.abspath("__file__")))
sys.path.insert(0, ".")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from src.background_classification import classify_background

DATA_DIR = "iapr-26-uno-vision-challenge/train_images"

# Pick two images that illustrate the two background types
examples = ["L1000770", "L1000780"]

fig, axes = plt.subplots(1, len(examples), figsize=(14, 6))
for ax, image_id in zip(axes, examples):
    img_bgr = cv2.imread(os.path.join(DATA_DIR, image_id + ".jpg"))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    H, W   = img_rgb.shape[:2]
    ps     = 50

    label  = classify_background(img_rgb)

    ax.imshow(img_rgb)
    ax.set_title(f"{image_id}  →  background = '{label}'", fontsize=11)

    # Overlay the sampled corner patches
    for y0, x0 in [(0, 0), (0, W - ps), (H - ps, 0), (H - ps, W - ps)]:
        rect = mpatches.Rectangle((x0, y0), ps, ps,
                                   linewidth=2, edgecolor='red', facecolor='none')
        ax.add_patch(rect)
    ax.axis("off")

fig.suptitle("Background classification — red boxes = sampled corner patches", fontsize=13)
plt.tight_layout()
plt.show()

### A.2 Active Player Detection (`src/active_player_detection.py`)

An active-player token is placed next to the hand of the player whose turn it is.
The token type depends on the background (see A.1):

| Background | Token shape | Token colour |
|------------|-------------|--------------|
| `white` | Rectangle | Black |
| `noisy` | Circle | Yellow |

The pipeline has four stages.

---

#### Stage 1 — HSV Thresholding

We convert the image to HSV and apply `cv2.inRange` with colour-specific bounds:

| Token | H range | S range | V range | Rationale |
|-------|---------|---------|---------|----------|
| Black | 0–180° (any) | 0–75 | 0–140 | Low saturation, dark value isolates the black tile |
| Yellow | 18–30° | 70–255 | 120–255 | Narrow hue avoids lime-green leaves (H ≥ 31) |

The **yellow upper bound H ≤ 30** is the critical design choice: pixel analysis of token and leaf crops showed that token pixels cluster at H ∈ [22, 33] while leaf pixels start at H ≈ 32. Cutting at 30 retains ~87 % of token pixels while excluding virtually all leaf pixels.

The **black upper bound V ≤ 140** removes the white card borders that would otherwise leak into the dark mask (white pixels have V ≈ 255).

---

#### Stage 2 — Morphological Opening

Two structuring elements are used:

| Token | Kernel | Effect |
|-------|--------|--------|
| Yellow | Ellipse 25 × 25 | Erodes thin arcs from skip-card symbols that share the yellow hue |
| Black | Ellipse 5 × 5 | Light denoising only (black tile is already compact) |

The large ellipse for the yellow token is sized to destroy the skip-card arc (thin, ~10 px wide) while preserving the solid token disc (~50 px radius).

---

#### Stage 3 — Blob Selection

For each contour surviving the opening, we compute a shape score and accept only blobs in the area range **[2 000, 80 000] px²**:

| Token | Shape score | Formula | Threshold |
|-------|-------------|---------|----------|
| Yellow | Circularity | $\frac{4\pi A}{P^2}$ | ≥ 0.4 |
| Black | Extent | $\frac{A}{A_{\text{bbox}}}$  | ≥ 0.4 |

The largest qualifying blob is taken as the token.

---

#### Stage 4 — Centroid-to-Player Mapping

The image is divided into four player zones using fraction-of-image thresholds:

```
       ┌──────────────────────────────┐
       │            p3 (top)          │
       │  p4    ┌──────────┐    p2   │
       │ (left) │  center  │ (right) │
       │        └──────────┘         │
       │           p1 (bottom)        │
       └──────────────────────────────┘
```

Rules (evaluated in order, short-circuit):
1. `user_y < H/5` and `W/4 < cx < 3W/4` → **p1** (bottom)
2. `user_y > 4H/5` and `W/4 < cx < 3W/4` → **p3** (top)
3. `cx > 3.85·W/5` → **p2** (right)
4. `cx < 1.15·W/5` → **p4** (left)
5. Fallback: nearest edge by pixel distance

(`user_y = H − cy` converts from image coordinates, where y increases downward, to user-facing coordinates where y increases upward.)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from src.background_classification import classify_background
from src.active_player_detection  import detect_active_player, _BLACK_TOKEN_HSV, _YELLOW_TOKEN_HSV

DATA_DIR = "iapr-26-uno-vision-challenge/train_images"
IMAGE_ID = "L1000770"

img_bgr = cv2.imread(os.path.join(DATA_DIR, IMAGE_ID + ".jpg"))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
hsv     = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)

background = classify_background(img_rgb)
active     = detect_active_player(img_rgb, background)
print(f"Background: {background}   →   Active player: {active}")

# Show the token mask before and after opening
if background == 'white':
    raw_mask = cv2.inRange(hsv, _BLACK_TOKEN_HSV['lower'], _BLACK_TOKEN_HSV['upper'])
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
else:
    raw_mask = cv2.inRange(hsv, _YELLOW_TOKEN_HSV['lower'], _YELLOW_TOKEN_HSV['upper'])
    kernel   = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))

opened_mask = cv2.morphologyEx(raw_mask, cv2.MORPH_OPEN, kernel)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(img_rgb);                     axes[0].set_title(f"Input ({IMAGE_ID})")
axes[1].imshow(raw_mask,    cmap='gray');    axes[1].set_title("Raw HSV mask")
axes[2].imshow(opened_mask, cmap='gray');    axes[2].set_title("After morphological opening")
for ax in axes: ax.axis('off')
plt.suptitle(f"Active player detection — background='{background}', detected='{active}'", fontsize=13)
plt.tight_layout()
plt.show()

### A.3 Background Removal

*Content to be added.*

### A.4 Image Division

*Content to be added.*

### A.5 Card Detection (`src/card_detection.py`)

Detection transforms a BGR region (one of the five areas produced in A.4) into a list of `(card_image, color)` pairs.
It consists of three stages: mask preprocessing, quadrilateral detection, and image extraction.

#### A.5.1 HSV Segmentation — `preprocess_mask`

UNO cards stand out from the background by their vivid colour. We work in HSV space
rather than BGR because the hue channel H is robust to lighting variations.

Six colour ranges are defined:

| Colour | Hue (H) | Note |
|--------|---------|------|
| Red lo | 0 – 8° | Red wraps around H = 0 in HSV |
| Red hi | 165 – 179° | Complement of red |
| Yellow | 22 – 36° | |
| Green | 45 – 85° | |
| Blue | 90 – 130° | |
| Dark (wild/+4) | S ≤ 100, V ≤ 120 | Black body of joker cards |

The saturation `sat_lo` and value `val_lo` lower bounds for coloured ranges are tunable
parameters to adapt to lighting conditions.

**Morphological pipeline applied to each mask:**
1. `cv2.inRange` → raw binary mask
2. `MORPH_OPEN` (kernel ≈ 1.7 % of region size) → removes small noise blobs
3. `MORPH_CLOSE` (kernel ≈ 10 % of region size) → fills internal gaps inside the card
4. `_fill_holes`: flood-fill from the image border closes open contours

**Suppression of coloured wedges inside joker cards:**  
Wild/+4 cards have a dark body but coloured sectors at the centre.
After hole-filling, the union of all dark masks covers the full joker body.
This filled dark mask is subtracted from every coloured mask
(`bitwise_and(colored, NOT(dark))`) so the wedges no longer produce false blobs
in the coloured masks.

In [ ]:
import os, sys
os.chdir(os.path.dirname(os.path.abspath("__file__")))
sys.path.insert(0, ".")

import cv2
import matplotlib.pyplot as plt
from src.card_detection import preprocess_mask, debug_mask

DATA_DIR = "iapr-26-uno-vision-challenge/train_images"
IMAGE_ID = "L1000770"

img = cv2.imread(os.path.join(DATA_DIR, IMAGE_ID + ".jpg"))
H, W = img.shape[:2]

# Centre region (middle third in both axes)
region = img[H//3 : 2*H//3, W//3 : 2*W//3]

mask_vis = debug_mask(region, name="center", sat_lo=80, val_lo=120)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(region, cv2.COLOR_BGR2RGB))
axes[0].set_title("Centre region")
axes[0].axis("off")
axes[1].imshow(cv2.cvtColor(mask_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title("Colour masks (red=red, cyan=yellow, green=green, blue=blue, gray=dark)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

#### A.5.2 Quad Detection — `get_quads`

From the masks, we search for rectangles corresponding to cards.
Three complementary methods are used in order of decreasing reliability.

**Pre-processing:** red-lo and red-hi masks are merged (`bitwise_or`) before processing.
This prevents the same red card from being detected twice (once per range).

---

##### Method 1 — Direct detection (M1)

For each contour in a mask:
- `cv2.minAreaRect` → minimum-area oriented rectangle
- Criterion: **area ≥ threshold** (15 % of the minimum region dimension)² AND **side ratio** within ±30 % of the UNO card ratio (87/56 ≈ 1.55)
- If both criteria are satisfied → the rectangle is directly accepted as a quad

This method works for fully visible cards with a compact mask.

---

##### Method 2 — Fragment pairing (M2)

When a card is partially hidden or its mask splits into multiple blobs, M2 tries to
**combine fragments of the same colour**:

- Coloured cards: all **pairs** of same-colour blobs are tested
- Dark cards: **all subsets** of dark blobs are tested (they can fragment into 3+)
- For each combination: `minAreaRect` on the union of points → aspect ratio check
- Score: distance to the reference area (derived from M1 cards already found)
- Greedy selection: each blob is used at most once

---

##### Method 3 — Edge-based reconstruction (M3)

For orphan blobs (not claimed by M1 or M2), the quadrilateral is reconstructed from a
**visible 90° corner on the blob's convex hull**:

1. `cv2.convexHull` + `approxPolyDP` on the blob points
2. For each consecutive triplet of hull vertices (A, B, C) with an angle at B ≈ 90°:
   - Unit vector `u = (A − B) / |A − B|` (direction of edge BA)
   - Exact perpendicular `v` oriented toward C (±90° rotation of u)
   - Optional confirmation: search for a 3rd hull edge parallel to u or v
     at distance ≈ card width or height → disambiguates which of u, v is the width
   - Fixed-size card rectangle (scaled from 140 × 218) anchored at corner B
3. Acceptance: ≥ 90 % of the blob covered by the proposed quad
4. Score: minimum mean distance from quad edges to the blob boundary
5. Deduplication by IoU > 50 %

The fixed card size is extrapolated from the median area of M1+M2 candidates
(`ref_w = sqrt(ref_area / aspect_ratio)`).

In [ ]:
from src.card_detection import debug_quads

vis = debug_quads(region, sat_lo=80, val_lo=120)

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Detected quads — M1 (direct), M2 (pairing), M3 (reconstruction)")
plt.axis("off")
plt.tight_layout()
plt.show()

#### A.5.3 Image Extraction — `get_card_images`

For each candidate quad:

1. **Coverage filter** (coloured cards only): at least 60 % of the quad's interior must be
   covered by detected pixels (union of all masks). This rejects reconstructed quads
   that fall mostly on background.

2. **Perspective transform** (`_warp_card`):
   - The quad is **expanded by 25 %** from its centroid to capture the card borders
   - `cv2.getPerspectiveTransform` + `cv2.warpPerspective` → rectified image
   - If width > height, rotate 90° to enforce portrait orientation
   - **Tight crop** (skipped for dark cards): detects coloured pixels (S > 25 or V > 180)
     within the inner 7.5 % margin to locate the actual card edge and trim white padding
   - Final resize: **140 × 218 pixels** (canonical format)

3. **Size filter**: if ≥ 2 candidates, quads whose area deviates more than 50 % from the
   reference (`_CARD_REF_AREA_PX = 140 000 px`) are rejected.
   Tolerance is doubled for dark cards (fragmented blobs → noisier area estimates).

4. **Non-Maximum Suppression (NMS)**: sort by bounding-box area descending;
   for each quad, suppress if IoU > 50 % with an already-kept quad **of the same colour**.
   Additionally, any coloured quad whose centroid falls inside a kept dark quad is suppressed
   (prevents a joker corner from being detected as a coloured card).

In [ ]:
from src.card_detection import detect_cards

cards = detect_cards(region, sat_lo=80, val_lo=120)
print(f"{len(cards)} card(s) detected")

if cards:
    fig, axes = plt.subplots(1, len(cards), figsize=(4 * len(cards), 5))
    if len(cards) == 1:
        axes = [axes]
    for ax, (card_img, color) in zip(axes, cards):
        ax.imshow(cv2.cvtColor(card_img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"color: {color}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

### A.6 Card Classification (`src/card_classification.py`)

Once each card is extracted as a 140 × 218 image, we identify its **value**
(0–9, skip, reverse, draw_2, wild, draw_4) to form the full label (e.g. `r_5`, `b_skip`).

The method relies on **corner patch matching**: each UNO card prints its value
as white ink symbols on the coloured background in the top-left and bottom-right corners.

#### A.6.1 Corner Extraction — `_extract_corner`

Each UNO card displays its value in the top-left (TL) corner and in the bottom-right
(BR) corner upside-down. Both corners are exploited to handle partially hidden cards.

**TL corner**: patch `img[7:55, 7:40]` → size **48 × 33 px**  
**BR corner**: patch `img[H-55:H-7, W-40:W-7]`, rotated 180° to align its orientation with TL

The 7-pixel white card border is skipped (`_BORDER = 7`) to retain only the printed symbol.

**Binarization**: a pixel is considered "white" if all its BGR channels exceed 210
(`_WHITE_THRESH = 210`). This produces a float32 binary mask (1.0 = white ink, 0.0 = coloured
background), capturing the symbol independently of the background colour.

#### A.6.2 Template Building — `build_templates_from_labeled`

Templates are built from images manually labeled using `label_cards.py`.

Directory layout:
```
labeled_cards/
    0/        ← all card images with value 0
    1/
    ...
    skip/
    reverse/
    draw_2/
    wild/
    draw_4/
```

For each value:
1. Each image is resized to 140 × 218
2. Both the TL mask **and** the BR mask are extracted (two observations per image)
3. The final template is the **mean** of all masks, equally weighted

Averaging smooths out lighting and printing variations between images.
The resulting template looks like a blurred version of the printed corner symbol.

In [ ]:
import numpy as np
from src.card_classification import load_templates

templates = load_templates("templates")
print(f"Available values: {sorted(templates.keys())}")

values = sorted(templates.keys())
n = len(values)
fig, axes = plt.subplots(1, n, figsize=(2.5 * n, 3))
for ax, val in zip(axes, values):
    ax.imshow(templates[val], cmap="gray", vmin=0, vmax=1)
    ax.set_title(val, fontsize=9)
    ax.axis("off")
fig.suptitle("Corner templates (mean white-ink mask over all labeled images)", fontsize=12)
plt.tight_layout()
plt.show()

#### A.6.3 Classification — `classify_card`

Classification uses `cv2.matchTemplate` with the **TM_CCOEFF_NORMED** metric
(normalised cross-correlation), which returns a score in [−1, 1] robust to
global brightness variations.

Since the template and the patch have the same size (48 × 33), `matchTemplate` returns
a (1, 1) array; the scalar score is `result[0, 0]`.

**Dark cards** (`color == 'dark'`, wild / +4):
- Only the `wild` and `draw_4` templates compete
- If both are absent: density heuristic on white pixels in TL
  (draw_4 prints "+4" → denser than the wild circle)

**Coloured cards**:
- Both TL **and** BR patches are extracted from the detected card
- For each candidate value:
  `score(v) = max( matchTemplate(TL, tmpl[v]), matchTemplate(BR, tmpl[v]) )`
- The winning value is the argmax of the scores
- The final label is `{color_prefix}_{value}`:
  `r_5`, `y_skip`, `g_draw_2`, `b_reverse`, `wild`, `draw_4`

Taking the max over TL and BR allows correct classification of partially hidden cards:
if TL is obstructed, BR provides a reliable score.

In [ ]:
from src.card_classification import classify_card

cards = detect_cards(region, sat_lo=80, val_lo=120)
print(f"{len(cards)} card(s) detected")

if cards:
    fig, axes = plt.subplots(1, len(cards), figsize=(4 * len(cards), 5))
    if len(cards) == 1:
        axes = [axes]
    for ax, (card_img, color) in zip(axes, cards):
        label = classify_card(card_img, color, templates)
        ax.imshow(cv2.cvtColor(card_img, cv2.COLOR_BGR2RGB))
        ax.set_title(label, fontsize=13)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

#### A.6.4 Global Evaluation

The script `test_card_classification.py` evaluates classification on the full training dataset
and reports accuracy per region (players 1–4 and centre):
- **count accuracy**: fraction of images where the number of detected cards is correct
- **label accuracy**: fraction of images where both the count and the labels are correct (after sorting)

For failing images, a figure is shown with the region, the mask, and the detected cards
annotated with their predicted labels.

In [ ]:
# To run the full evaluation (shows only failures):
# !python test_card_classification.py

# For a specific image:
# !python test_card_classification.py L1000770

---
## Part B — Results and Discussion

*Content to be added.*